# Python Algorithms and Movie Analytics

This notebook demonstrates a progression from compact algorithmic problems to
memory-conscious analysis of movie metadata and rating records. All reusable
logic lives in `src/`; the notebook focuses on examples, results and
interpretation.

The included files are fabricated samples. If the complete files are placed in
`data/raw/`, the notebook selects them automatically.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory.")

sys.path.insert(0, str(PROJECT_ROOT))

from src.algorithms import (
    check_preceding_sum,
    enumerate_animal_combinations,
    find_matches,
    top_bigrams,
)
from src.movie_analytics import (
    best_french_exclusive_movie,
    earliest_rating,
    most_rated_movie,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
SAMPLE_DIR = PROJECT_ROOT / "data" / "sample"
required_files = ("linuxwords", "movies.json", "ratings.csv")
DATA_DIR = RAW_DIR if all((RAW_DIR / name).exists() for name in required_files) else SAMPLE_DIR

words_path = DATA_DIR / "linuxwords"
movies_path = DATA_DIR / "movies.json"
ratings_path = DATA_DIR / "ratings.csv"
print(f"Using {'complete' if DATA_DIR == RAW_DIR else 'sample'} data from {DATA_DIR.relative_to(PROJECT_ROOT)}")

Using sample data from data/sample


## 1. Sliding-window sequence validation

Each value after the initial window should equal the sum of the preceding
`n` values. The examples include both valid and invalid sequences.

In [2]:
sequence_examples = [
    ([1, 2, 3, 5, 8, 13], 2),
    ([1, 2, 3, 5, 8, 13], 3),
    ([1, 1, 2, 2, 6, 11, 21], 4),
]

for values, window in sequence_examples:
    print(f"window={window}: {check_preceding_sum(values, window)} — {values}")

window=2: True — [1, 2, 3, 5, 8, 13]
window=3: False — [1, 2, 3, 5, 8, 13]
window=4: True — [1, 1, 2, 2, 6, 11, 21]


## 2. Character-bigram frequency

Words are cleaned individually, so adjacent characters are counted within a
word but never across line boundaries. Frequency ties are resolved
alphabetically for reproducibility.

In [3]:
with words_path.open(encoding="utf-8", errors="ignore") as source:
    leading_bigrams = top_bigrams(source, limit=10)

for rank, (bigram, count) in enumerate(leading_bigrams, start=1):
    print(f"{rank:>2}. {bigram}: {count:,}")

 1. al: 2
 2. at: 2
 3. en: 2
 4. ie: 2
 5. im: 2
 6. nc: 2
 7. ti: 2
 8. an: 1
 9. ce: 1
10. ch: 1


## 3. Constraint-based animal combinations

The search finds non-negative counts of parrots, lions, ladybugs and scorpions
that satisfy both an animal total and a leg total. Two counts are enumerated;
the remaining two are solved algebraically.

In [4]:
toy_combinations = enumerate_animal_combinations(3, 10)
for combination in toy_combinations:
    print(combination)

larger_combinations = enumerate_animal_combinations(20, 146)
print(f"\n20 animals and 146 legs: {len(larger_combinations)} valid combinations")
for combination in larger_combinations[:5]:
    print(combination)

AnimalCombination(parrots=1, lions=2, ladybugs=0, scorpions=0)
AnimalCombination(parrots=2, lions=0, ladybugs=1, scorpions=0)

20 animals and 146 legs: 8 valid combinations
AnimalCombination(parrots=0, lions=0, ladybugs=7, scorpions=13)
AnimalCombination(parrots=0, lions=1, ladybugs=5, scorpions=14)
AnimalCombination(parrots=0, lions=2, ladybugs=3, scorpions=15)
AnimalCombination(parrots=0, lions=3, ladybugs=1, scorpions=16)
AnimalCombination(parrots=1, lions=0, ladybugs=4, scorpions=15)


## 4. Match-three board search

The board is searched horizontally, vertically and along both diagonal
directions. Endpoint checks include the final legal starting row and column,
avoiding a common off-by-one error.

In [5]:
board = [
    ["x", "o", "x", "+", "x", "-", "+", "o", "+"],
    ["-", "+", "o", "-", "*", "x", "o", "x", "x"],
    ["-", "*", "o", "-", "o", "+", "*", "x", "+"],
    ["x", "+", "x", "o", "+", "*", "*", "+", "x"],
    ["+", "*", "*", "o", "o", "x", "o", "+", "x"],
    ["*", "-", "x", "x", "*", "-", "*", "x", "o"],
    ["*", "o", "x", "-", "o", "-", "x", "o", "*"],
    ["x", "+", "*", "*", "-", "o", "-", "+", "+"],
    ["-", "*", "x", "*", "*", "o", "+", "x", "*"],
]

matches = find_matches(board)
print(f"Detected {len(matches)} matches:")
for match in matches:
    print(match)

Detected 2 matches:
((2, 2), (3, 3), (4, 4))
((4, 8), (5, 7), (6, 6))


## 5. Streaming movie-rating analysis

The CSV reader yields one rating at a time. The language-segment analysis uses
two passes: first to classify users, then to aggregate ratings for users whose
matched movies are all labelled French.

In [6]:
earliest = earliest_rating(ratings_path)
most_rated = most_rated_movie(movies_path, ratings_path)
best_french = best_french_exclusive_movie(movies_path, ratings_path)

print(f"Earliest rating: user {earliest.user_id}, timestamp {earliest.timestamp}")
print(f"Most rated: {most_rated.title} — {most_rated.rating_count:,} ratings")
print(
    "Best among French-exclusive raters: "
    f"{best_french.title} — {best_french.average_rating:.3f} "
    f"from {best_french.rating_count} ratings"
)

Earliest rating: user u6, timestamp 90
Most rated: Le Rivage — 5 ratings
Best among French-exclusive raters: Le Rivage — 4.625 from 4 ratings


## Interpretation

The sample outputs verify behaviour and reproducibility, but they are not real
findings. With the complete files, the recorded analysis identifies user 5274
as the earliest rater, *The Million Dollar Hotel* as the most-rated title and
*Sous le Sable* as the highest-rated qualifying title among French-exclusive
raters.

These results depend on the supplied identifier mapping and language labels.
They describe the dataset only and should not be interpreted as current or
population-level movie preferences.